# Arabic corpus analysis: frequencies, keyness and KWIC

A step-by-step walkthrough for researchers new to computational text analysis. Every text in this notebook is **invented for teaching**, so nothing here is evidence about real Arabic discourse.

You will:
1. normalise and tokenise Arabic text, and see what those choices change;
2. count word frequencies in two small corpora;
3. calculate one keyness score (G²) by hand;
4. compare every word type across the corpora; and
5. read words in context (KWIC).

The command-line version and a fuller discussion of the methods are in [the tutorial README](https://github.com/s1112124/digital-research-methods/tree/main/tutorials/arabic-corpus-analysis).

## 1. Get the files

In Google Colab, the next cell downloads the public repository. On your own computer, run this notebook from the `tutorials/arabic-corpus-analysis` folder and it will use the local files instead.

In [1]:
from pathlib import Path
import subprocess
import sys

tutorial = Path.cwd()
if not (tutorial / 'data' / 'focus.csv').is_file():
    repo = Path('/content/digital-research-methods')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/s1112124/digital-research-methods.git', str(repo)],
                       check=True)
    tutorial = repo / 'tutorials' / 'arabic-corpus-analysis'

sys.path.insert(0, str(tutorial))
import analysis  # the tutorial's own, tested functions

print('Corpus files found:', sorted(p.name for p in (tutorial / 'data').glob('*.csv')))

Corpus files found: ['focus.csv', 'reference.csv']


## 2. Read the two corpora

The **focus** corpus contains 12 invented sentences about digital research workshops; the **reference** corpus contains 12 about archives and reading. Keyness compares one against the other, so the choice of reference corpus shapes every result.

In [2]:
focus_docs = analysis.read_documents(tutorial / 'data' / 'focus.csv')
reference_docs = analysis.read_documents(tutorial / 'data' / 'reference.csv')

print(len(focus_docs), 'focus documents |', len(reference_docs), 'reference documents')
print('F01:', focus_docs[0]['text'])
print('R01:', reference_docs[0]['text'])

12 focus documents | 12 reference documents
F01: في ورشة البيانات يتعلم المشاركون تنظيف الجداول وتحليل النتائج باستخدام أدوات رقمية.
R01: في الأرشيف يقرأ الباحث الوثائق القديمة ويكتب ملاحظات عن السياق.


## 3. Normalise and tokenise

Tools built for English often split words on spaces and stop there. Arabic needs explicit decisions first. This tutorial:

* removes short-vowel diacritics (tashkeel) and the elongation character tatweel (ـ);
* keeps the different spellings of alef (آ أ إ ٱ) **distinct by default**, because merging them can erase meaningful differences; and
* treats each run of Arabic letters as one surface word, so it does **not** separate attached prefixes such as و or ال.

The invented example below has diacritics, tatweel and hamza-on-alef spellings. Compare the last two lines to see what optional alef folding changes.

In [3]:
example = 'إِنَّ البيانـــاتِ أكثرُ من أرقامٍ'

print('Original:   ', example)
print('Normalised: ', analysis.normalize_arabic(example))
print('Tokens:     ', analysis.tokenize(example))
print('Alef folded:', analysis.tokenize(example, fold_alef=True))

Original:    إِنَّ البيانـــاتِ أكثرُ من أرقامٍ
Normalised:  إن البيانات أكثر من أرقام
Tokens:      ['إن', 'البيانات', 'أكثر', 'من', 'أرقام']
Alef folded: ['ان', 'البيانات', 'اكثر', 'من', 'ارقام']


## 4. Count word frequencies

Raw counts depend on corpus size, so we also report each word's rate per 10,000 tokens. Only relative rates can be compared across corpora of different sizes.

In [4]:
import pandas as pd

focus_counts = analysis.count_tokens(focus_docs)
reference_counts = analysis.count_tokens(reference_docs)
total_f = sum(focus_counts.values())
total_r = sum(reference_counts.values())
print('Focus tokens:', total_f, '| Reference tokens:', total_r)

pd.DataFrame(analysis.frequency_rows(focus_counts)).head(10)

Focus tokens: 108 | Reference tokens: 97


,word,count,per_10000
0,البيانات,7,648.15
1,النتائج,3,277.78
2,في,3,277.78
3,أدوات,2,185.19
4,البرمجة,2,185.19
5,التحليل,2,185.19
6,التدريب,2,185.19
7,الجداول,2,185.19
8,الرسوم,2,185.19
9,الطلاب,2,185.19


## 5. One keyness score by hand

Keyness asks whether a word is used more often in one corpus than we would expect if both corpora used it at the same rate. For one word, we build a 2×2 table of **observed** counts (O), work out the **expected** counts (E) if the rates were equal, and combine them as

**G² = 2 Σ O ln(O / E)**

Cells with an observed count of zero contribute nothing to the sum.

In [5]:
import math

word = 'البيانات'
a, b = focus_counts[word], reference_counts[word]   # the word
c, d = total_f - a, total_r - b                     # every other token
N = total_f + total_r

observed = pd.DataFrame({'focus': [a, c], 'reference': [b, d]},
                        index=[word, 'all other words'])
expected = pd.DataFrame({'focus': [total_f * (a + b) / N, total_f * (c + d) / N],
                         'reference': [total_r * (a + b) / N, total_r * (c + d) / N]},
                        index=observed.index)
display(observed)
display(expected.round(2))

g2 = 2 * sum(o * math.log(o / e)
             for o, e in zip(observed.values.ravel(), expected.values.ravel()) if o > 0)
print(f'G² by hand:              {g2:.3f}')
print(f'G² from analysis.py:     {analysis.likelihood_ratio(a, total_f, b, total_r):.3f}')

,focus,reference
البيانات,7,0
all other words,101,97


,focus,reference
البيانات,3.69,3.31
all other words,104.31,93.69


G² by hand:              9.194
G² from analysis.py:     9.194


## 6. Keyness for every word type

The tutorial code repeats that calculation for every word and adds:

* an **effect size** (`log2_rate_ratio_adjusted`): how much more often, and in which direction, with 0.5 added to each count so that words absent from one corpus do not produce infinite values;
* an approximate **p-value**, and a **Benjamini–Hochberg q-value** that adjusts for testing many words at once.

With corpora this small and this deliberately different, treat the numbers as a demonstration of the mechanics, not as findings. A high G² is not the same as an important difference: compare the G² ranking with the effect sizes.

In [6]:
keyness = pd.DataFrame(analysis.keyness_rows(focus_counts, reference_counts))
keyness[['word', 'focus_count', 'reference_count', 'g2',
         'log2_rate_ratio_adjusted', 'bh_q_exploratory', 'direction']].head(10)

,word,focus_count,reference_count,g2,log2_rate_ratio_adjusted,bh_q_exploratory,direction
0,البيانات,7,0,9.19435,3.7527,0.272266,focus
1,على,0,4,6.07509,-3.3241,0.272266,reference
2,الأرشيف,0,3,4.53943,-2.9616,0.272266,reference
3,النتائج,3,0,3.88527,2.6531,0.272266,focus
4,السياق,0,2,3.01514,-2.4761,0.272266,reference
5,القديمة,0,2,3.01514,-2.4761,0.272266,reference
6,المكتبة,0,2,3.01514,-2.4761,0.272266,reference
7,الوثائق,0,2,3.01514,-2.4761,0.272266,reference
8,سجلات,0,2,3.01514,-2.4761,0.272266,reference
9,أدوات,2,0,2.58121,2.1677,0.272266,focus


## 7. Keyword in context (KWIC)

Frequencies tell you *how often*; concordance lines show *how* a word is used. KWIC tools designed for left-to-right scripts call the words before the search term its "left" context. In Arabic those words sit to the **right** of the term on the page, so the table below labels the columns **preceding** and **following** instead.

In [7]:
kwic = pd.DataFrame(analysis.kwic_rows(focus_docs, 'البيانات', corpus_label='focus', window=4))
kwic.rename(columns={'left': 'preceding', 'right': 'following'})[['doc_id', 'preceding', 'node', 'following']]

,doc_id,preceding,node,following
0,F01,في ورشة,البيانات,يتعلم المشاركون تنظيف الجداول
1,F02,يشرح المدرب طريقة جمع,البيانات,وتنظيم الملفات قبل بناء
2,F03,تتيح البرمجة للباحثين معالجة,البيانات,وتوثيق خطوات العمل
3,F04,راجع الطلاب,البيانات,وحددوا القيم الناقصة في
4,F06,ناقش الفريق جودة,البيانات,ومصادر الأخطاء أثناء التدريب
5,F09,المجموعة البرمجة لقراءة ملف,البيانات,وتحويل الأعمدة
6,F10,بسؤال بحثي ثم اختيار,البيانات,الملائمة


## 8. Try it yourself

Change `query` to another single word (for example الوثائق), or set `fold` to `True`, and run the cell again. Which results change, and why?

In [8]:
query = 'الوثائق'   # one Arabic word
fold = False         # True merges آ أ إ ٱ with ا

rows = analysis.kwic_rows(focus_docs + reference_docs, query,
                          corpus_label='both', window=4, fold_alef=fold)
pd.DataFrame(rows).rename(columns={'left': 'preceding', 'right': 'following'})[['doc_id', 'preceding', 'node', 'following']]

,doc_id,preceding,node,following
0,R01,في الأرشيف يقرأ الباحث,الوثائق,القديمة ويكتب ملاحظات عن
1,R02,القراء على العثور على,الوثائق,والكتب النادرة


## 9. Optional: run the automated tests

The tutorial's functions are covered by tests that check tokenisation, the G² calculation and KWIC behaviour.

In [9]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'],
                        cwd=tutorial, capture_output=True, text=True)
print(result.stderr.strip().splitlines()[-1])

OK


## Reflection

* What happens to the keyness ranking if you add an invented document to the focus corpus?
* Does an apparent keyword difference here demonstrate anything about real Arabic discourse? Why not?
* How might a different tokeniser, reference corpus, sampling strategy or alef-folding decision change the ranking?

**Data:** all texts are invented for this tutorial. See the README for the tokenisation decisions, statistical assumptions and guidance on interpretation.